# 10 — Advanced

Scheduler, fallback LLM chain, background audio, noise filter, custom STT/TTS, custom LLM HTTP.


## Prerequisites

| Tier | Cells | Required env |
|------|-------|--------------|
| T1+T2 (§1) | always | _none_ |
| T3 (§2) | per-cell | provider keys auto-detected |
| T4 (§3) | gated | `ENABLE_LIVE_CALLS=1` + carrier creds |


In [ ]:
%load_ext autoreload
%autoreload 2

import _setup
env = _setup.load()
print(f'getpatter version target: {env.patter_version}')


## §1: Quickstart

Runs end-to-end with zero API keys.


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline (no network, no carrier calls).


In [ ]:
import sys
import getpatter
with _setup.cell('version_check', tier=1, env=env) as ok:
    if ok:
        print(f'getpatter {getpatter.__version__} on Python {sys.version.split()[0]}')
        assert getpatter.__version__ >= env.patter_version, \
            f'installed {getpatter.__version__} < target {env.patter_version}'


### Local mode
Construct a Patter instance with a Twilio carrier. No API key — runs entirely on your machine.


In [ ]:
from getpatter import Patter, Twilio
with _setup.cell('local_mode', tier=1, env=env) as ok:
    if ok:
        p = Patter(
            carrier=Twilio(
                account_sid='ACtest00000000000000000000000000',
                auth_token='test',
            ),
            phone_number='+15555550100',
            webhook_url='https://example.com/webhook',
        )
        # Verify carrier is wired up via LocalConfig.
        assert p._local_config.telephony_provider == 'twilio'
        assert p._local_config.phone_number == '+15555550100'
        print(f'provider={p._local_config.telephony_provider}  phone={p._local_config.phone_number}')


### Cloud mode (coming soon)
When `api_key=` is supported, Patter cloud handles telephony. For now, the SDK raises `NotImplementedError` — this cell verifies the guard.


In [ ]:
from getpatter import Patter
with _setup.cell('cloud_mode', tier=1, env=env) as ok:
    if ok:
        try:
            Patter(api_key='pt_test_xxx')
            raise AssertionError('expected NotImplementedError — cloud mode guard missing')
        except NotImplementedError as exc:
            print(f'cloud mode guard OK: {exc}')


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline* (STT + LLM + TTS). The factory derives the engine from `engine=` / `stt=`/`tts=`.


In [ ]:
from getpatter import Patter, Twilio, OpenAIRealtime, ElevenLabsConvAI
with _setup.cell('agent_engines', tier=1, env=env) as ok:
    if ok:
        p = Patter(
            carrier=Twilio(account_sid='ACtest00000000000000000000000000',
                           auth_token='test'),
            phone_number='+15555550100',
            webhook_url='https://example.com/webhook',
        )
        rt = p.agent(system_prompt='hi', engine=OpenAIRealtime(api_key='sk-test'))
        cv = p.agent(system_prompt='hi', engine=ElevenLabsConvAI(api_key='el-test', agent_id='a1'))
        pl = p.agent(system_prompt='hi')  # default: OpenAI Realtime fallback
        assert rt.provider == 'openai_realtime', rt.provider
        assert cv.provider == 'elevenlabs_convai', cv.provider
        assert pl.provider == 'openai_realtime', pl.provider
        print(f'rt.provider={rt.provider}  cv.provider={cv.provider}  pl.provider={pl.provider}')


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline (no network, no carrier calls).


### Local mode
Construct a Patter instance with a Twilio carrier. No API key — runs entirely on your machine.


### Cloud mode
Same SDK, just an `api_key=` instead of a carrier — Patter cloud handles telephony.


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline* (STT + LLM + TTS). The factory derives the mode from `engine=` / `stt=`/`tts=`.


## §2: Feature Tour

One cell per feature/provider. Missing keys auto-skip.


## §2 — Feature Tour

Exercises scheduler, IVR/DTMF, background audio, and the EventBus.


### Scheduler


In [ ]:
from datetime import datetime, timedelta, timezone
from getpatter import schedule_once, schedule_interval, schedule_cron, ScheduleHandle
with _setup.cell('scheduler', tier=1, env=env) as ok:
    if ok:
        fired: list[str] = []

        h1 = schedule_once(
            datetime.now(tz=timezone.utc) + timedelta(hours=1),
            lambda: fired.append('once'),
        )
        h2 = schedule_interval(300, lambda: fired.append('interval'))
        h3 = schedule_cron('0 9 * * 1-5', lambda: fired.append('cron'))

        print(f'once     job_id: {h1.job_id[:20]}...')
        print(f'interval job_id: {h2.job_id[:20]}...')
        print(f'cron     job_id: {h3.job_id[:20]}...')

        h1.cancel()
        h2.cancel()
        h3.cancel()
        print('All handles cancelled — no callbacks should fire')
        assert isinstance(h1, ScheduleHandle)


### IVR / DTMF


In [ ]:
from getpatter import DtmfEvent, format_dtmf
with _setup.cell('ivr_dtmf', tier=1, env=env) as ok:
    if ok:
        # Simulate caller pressing 1-800-555-0100 on their keypad.
        sequence = [
            DtmfEvent.ONE, DtmfEvent.EIGHT, DtmfEvent.ZERO, DtmfEvent.ZERO,
            DtmfEvent.FIVE, DtmfEvent.FIVE, DtmfEvent.FIVE,
            DtmfEvent.ZERO, DtmfEvent.ONE, DtmfEvent.ZERO, DtmfEvent.ZERO,
        ]
        formatted = format_dtmf(sequence)
        print(f'DTMF digits:    {[e.value for e in sequence]}')
        print(f'format_dtmf:    {formatted!r}')
        print(f'Available events: {[e.name for e in DtmfEvent]}')
        assert formatted == '1 8 0 0 5 5 5 0 1 0 0'


### Background audio


In [ ]:
from getpatter import BackgroundAudioPlayer, BuiltinAudioClip
with _setup.cell('background_audio', tier=1, env=env) as ok:
    if ok:
        print('Built-in audio clips:')
        for clip in BuiltinAudioClip:
            print(f'  {clip.name:20s} → {clip.value}')

        player = BackgroundAudioPlayer(
            source=BuiltinAudioClip.HOLD_MUSIC,
            volume=0.15,
            loop=True,
        )
        print(f'\nPlayer constructed: source={player.source}  volume={player.volume}  loop={player.loop}')
        assert player.volume == 0.15
        assert player.loop is True


### EventBus


In [ ]:
from getpatter import EventBus
with _setup.cell('event_bus', tier=1, env=env) as ok:
    if ok:
        bus = EventBus()
        received: list[dict] = []

        unsubscribe = bus.on('call_ended', lambda payload: received.append(payload))
        bus.emit('call_ended', {'call_id': 'c001', 'duration': 45})
        bus.emit('call_ended', {'call_id': 'c002', 'duration': 120})

        print(f'Received {len(received)} events:')
        for evt in received:
            print(f'  call_id={evt["call_id"]}  duration={evt["duration"]}s')

        unsubscribe()  # stop listening
        bus.emit('call_ended', {'call_id': 'c003'})
        print(f'After unsubscribe: still {len(received)} events (c003 not received)')
        assert len(received) == 2


## §3: Live Appendix

Real PSTN calls. Off by default — set `ENABLE_LIVE_CALLS=1`.
